# Feature Selection (BE, FS, RFE)

## Interview Revision Summary
- Goal: identify the most important predictors and remove weak, noisy, or redundant features.
- Backward Elimination: start with all features and remove the least significant ones using p-values.
- Forward Selection: start with no features and add features when they improve the model.
- RFE: use a model-based estimator to rank and eliminate features recursively.
- Data preparation matters: keep numeric columns, remove columns with too many missing values, and impute the remaining missing values.
- Interview tip: explain when you would choose statistical selection versus model-based selection depending on interpretability and complexity.


**P-value**

It is a massive misconception that data itself *has* to be normally distributed to calculate a $p$-value.

Here is the intuition of how $p$-values actually work when the world isn't a perfect bell curve, and how we choose the right tool for the job.

## 1. The Core Intuition: What is a $p$-value *really*?

A $p$-value does not care about the distribution of your **raw data**. It cares about the distribution of your **test statistic** (like a $t$-score, $F$-score, or $\chi^2$-score) *under the assumption that the Null Hypothesis ($H_0$) is true*.

Think of a $p$-value as a **"Surprise Index."**

1. You assume a boring world where your feature has zero effect ($H_0$).
2. You look at the mathematical pattern of what that boring world *should* look like.
3. You map your actual data's result onto that pattern.
4. The $p$-value is the probability of seeing a result as extreme as yours (or more extreme) just by pure, random luck.

If that probability is tiny (e.g., $p \lt 0.05$), you say, *"Wow, this is too weird to be a coincidence. The boring world assumption must be wrong. This feature is important."*

## 2. How are $p$-values calculated if data isn't Normal?

When your raw data is messy, non-normal, or skewed, statisticians use three main bridges to calculate $p$-values:

### Bridge A: The Central Limit Theorem (CLT)

This is the ultimate statistical superpower. The CLT states that if your sample size ($n$) is large enough (usually $n \gt 30$ is a loose rule of thumb), the **distribution of the sample means** will be normally distributed, *no matter what the underlying data looks like*.

Because regression coefficients in Backward Elimination are essentially specialized averages, as long as you have enough data points, the math naturally curves into a normal distribution at the macro level.

### Bridge B: Non-Parametric Distribution-Free Tests

If your sample size is tiny *and* your data is non-normal, parametric tests (like the standard $t$-test) break down. Enter **Non-Parametric statistics**. These methods don't assume a distribution shape. Instead of using the raw values, they often convert your data into **ranks** (1st place, 2nd place, 3rd place).

- **Example:** The *Wilcoxon Signed-Rank Test* or *Mann-Whitney U Test*.
- The $p$-value here is calculated based on combinatorics: *"What are the odds that all the high ranks randomly clustered into this one feature by pure chance?"*

### Bridge C: Permutation & Bootstrapping (Resampling)

If you don't want to rely on *any* theoretical math formulas, you can let your computer calculate a $p$-value using sheer brute force. This is called a **Permutation Test**.

1. Take your feature and your target variable.
2. Shuffle (permute) the target variable column randomly so that any real relationship between the feature and the target is intentionally destroyed.
3. Calculate the relationship score.
4. Repeat this 10,000 times to map out an empirical "boring world" distribution.
5. See where your *actual, unshuffled* data lands on this custom-built histogram.

## 3. Is there a single way to calculate it, or how do we choose?

There is **no single way**. The method is chosen based on a strict hierarchy of conditions. In machine learning libraries (like `statsmodels` in Python used for Backward Elimination), the algorithm makes this choice under the hood based on the model type you select.

Here is the general decision framework for how to choose:

| Data Condition | Best Mathematical Approach | Why? |
| --- | --- | --- |
| **Large Dataset, Continuous Data** | **Parametric ($t$-test / $F$-test)** | The Central Limit Theorem protects you; it's computationally fast and highly accurate. |
| **Small Dataset, Highly Skewed/Outliers** | **Non-Parametric (e.g., Rank-based)** | Protects the model from being pulled out of whack by a few extreme outlier numbers. |
| **Categorical Data (e.g., Yes/No)** | **Chi-Square ($\chi^2$) Test** | Calculates $p$-values based on counting frequencies instead of measuring distances/means. |
| **Complex/Unknown Distributions** | **Permutation / Bootstrapping** | Ideal when data breaks all traditional rules and you have the computing power to simulate the distribution. |

### How this maps back to Backward Elimination:

When you run Backward Elimination in regression, the software assumes your sample size is sufficient to use standard **$t$-tests** or **$F$-tests** for the $p$-values. If your data severely violates these assumptions (e.g., severe non-linearity), the $p$-values become inaccurate, which is exactly why Backward Elimination can make bad decisions on poorly prepared data!

Let's break down **Backward Elimination (BE)** completely.

## What is Backward Elimination?

Think of Backward Elimination as the "auditioning" process where everyone starts on stage, and you systematically vote off the weakest link one by one.

You begin with **all** possible features in your model. You then evaluate the performance of the model, identify the least significant feature, and drop it. You repeat this loop until every single feature left in the model meets a predefined standard of importance.

### The Step-by-Step Process:

1. **Choose a significance level:** Usually, a $p$-value threshold is set (e.g., $\alpha = 0.05$).
2. **Fit the full model:** Train your model using all available features.
3. **Identify the worst feature:** Look at the metric (like the highest $p$-value) for each feature.
4. **Make the cut:** If the worst feature's $p$-value is greater than your significance level ($p \gt \alpha$), remove it. If all features are below the threshold, your model is ready, and you stop.
5. **Rebuild and Repeat:** Fit the model again *without* the removed feature and go back to Step 3.

## The Statistical Concept Behind It

Backward Elimination relies heavily on **Hypothesis Testing**—specifically testing the null hypothesis for each feature's coefficient.

- **The Null Hypothesis ($H_0$):** The feature has *no effect* on the target variable (its coefficient is zero).
- **The Alternative Hypothesis ($H_1$):** The feature has a significant effect on the target variable.

When you look at the **$p$-value** of a feature, you are measuring the probability that the feature's relationship with the target happened by pure random chance.

- A **high $p$-value** (e.g., 0.45) means there is a 45% chance the feature is just noise. It fails to reject the null hypothesis, making it a prime candidate to be eliminated.
- Other metrics can be used instead of $p$-values, such as **AIC (Akaike Information Criterion)** or **BIC**, where you eliminate the feature whose removal results in the lowest (best) information criterion score.

## When to Use Backward Elimination

- **When you want highly interpretable models:** It's incredibly popular in linear and logistic regression because it leaves you with a statistically validated, lean set of features.
- **When you have a manageable number of features:** If your dataset has a modest number of features relative to the number of rows, BE works beautifully.
- **When feature interactions matter:** Because BE starts with *all* features in the pool, it captures the joint impact of features working together before throwing anything out. (This is a huge advantage over Forward Selection!).

## When NOT to Use It

- **High-Dimensional Data ($p \gt n$):** If you have more features than data points (e.g., genomic data with 10,000 features but only 100 patient samples), you physically cannot fit the initial "full model." BE fails right at Step 2.
- **Collinearity issues:** If two features are highly correlated, their individual $p$-values might both look terrible (high) initially, causing the algorithm to drop a feature that actually carries important signal.
- **Computationally heavy models:** If you are using massive datasets or algorithms that take hours to train (like deep neural networks), retraining the model over and over again for every single feature drop is computationally exhausting.

## Key Details & Pitfalls You Must Be Aware Of

- **The "Greedy" Nature:** Backward Elimination is a **greedy algorithm**. It makes the best local choice at each step. Once a feature is thrown out, it is gone forever. It will never test if adding that feature back later—in combination with a different subset—would have been magical.
- **Overfitting Hazard:** Because you are constantly making decisions based on the dataset's statistical quirks, your final model can sometimes overfit the training data. Standard practice dictates validating the final feature subset on a separate test set or via cross-validation.
- **Choosing $\alpha$ is arbitrary:** Setting your threshold at 0.05 is a standard convention, but it's not a magic law. Sometimes data scientists use a more generous $\alpha = 0.10$ during elimination to ensure they don't accidentally drop marginally useful features too early.

Ready to see how this compares to **Forward Selection**, or would you like to dive deeper into the code/math of how the $p$-value is calculated here?

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("D:/KrishNaikDataScience/LiveProjects/Level4_featureEngg/26_Apr_FeatureEngg_3/house-prices-advanced-regression-techniques/train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
# Cell 2: Data preparation for Backward Elimination
# Purpose: prepare a clean numeric dataset so regression-based feature selection can run safely.
# Sub-part 1: import statsmodels, which provides the OLS regression model used for p-value testing.
# Sub-part 2: keep only numeric columns because regression models usually cannot work directly with text or object data.
# Sub-part 3: measure missing values and identify columns with more than 30% missing data.
# Sub-part 4: drop those weak columns and inspect the cleaned table.

# 1. Backward Elimination (based on p-values in regression)

import statsmodels.api as sm

# Step 1: Select only numerical columns
df_numeric = df.select_dtypes(include=['number'])

# Step 2 & 3: Check for missing values and drop columns with >30% missing
total_rows = len(df_numeric)
missing_counts = df_numeric.isnull().sum() # a pandas Series with counts of missing values for each column
print("Missing counts:")
print(missing_counts)


Missing counts:
Id                 0
MSSubClass         0
LotFrontage      259
LotArea            0
OverallQual        0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
MasVnrArea         8
BsmtFinSF1         0
BsmtFinSF2         0
BsmtUnfSF          0
TotalBsmtSF        0
1stFlrSF           0
2ndFlrSF           0
LowQualFinSF       0
GrLivArea          0
BsmtFullBath       0
BsmtHalfBath       0
FullBath           0
HalfBath           0
BedroomAbvGr       0
KitchenAbvGr       0
TotRmsAbvGrd       0
Fireplaces         0
GarageYrBlt       81
GarageCars         0
GarageArea         0
WoodDeckSF         0
OpenPorchSF        0
EnclosedPorch      0
3SsnPorch          0
ScreenPorch        0
PoolArea           0
MiscVal            0
MoSold             0
YrSold             0
SalePrice          0
dtype: int64


In [4]:
missing_percentage = (missing_counts / total_rows) * 100
print("Missing percentages:")
print(missing_percentage) # pandas Series with percentage of missing values for each column

Missing percentages:
Id                0.000000
MSSubClass        0.000000
LotFrontage      17.739726
LotArea           0.000000
OverallQual       0.000000
OverallCond       0.000000
YearBuilt         0.000000
YearRemodAdd      0.000000
MasVnrArea        0.547945
BsmtFinSF1        0.000000
BsmtFinSF2        0.000000
BsmtUnfSF         0.000000
TotalBsmtSF       0.000000
1stFlrSF          0.000000
2ndFlrSF          0.000000
LowQualFinSF      0.000000
GrLivArea         0.000000
BsmtFullBath      0.000000
BsmtHalfBath      0.000000
FullBath          0.000000
HalfBath          0.000000
BedroomAbvGr      0.000000
KitchenAbvGr      0.000000
TotRmsAbvGrd      0.000000
Fireplaces        0.000000
GarageYrBlt       5.547945
GarageCars        0.000000
GarageArea        0.000000
WoodDeckSF        0.000000
OpenPorchSF       0.000000
EnclosedPorch     0.000000
3SsnPorch         0.000000
ScreenPorch       0.000000
PoolArea          0.000000
MiscVal           0.000000
MoSold            0.000000
YrSold 

In [5]:
columns_to_drop = missing_percentage[missing_percentage > 30].index.tolist()
df_cleaned = df_numeric.drop(columns=columns_to_drop)

print("Dropping columns with >30% missing values:")
print(columns_to_drop)
print("-" * 30)

df_cleaned.info()


Dropping columns with >30% missing values:
[]
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   OverallQual    1460 non-null   int64  
 5   OverallCond    1460 non-null   int64  
 6   YearBuilt      1460 non-null   int64  
 7   YearRemodAdd   1460 non-null   int64  
 8   MasVnrArea     1452 non-null   float64
 9   BsmtFinSF1     1460 non-null   int64  
 10  BsmtFinSF2     1460 non-null   int64  
 11  BsmtUnfSF      1460 non-null   int64  
 12  TotalBsmtSF    1460 non-null   int64  
 13  1stFlrSF       1460 non-null   int64  
 14  2ndFlrSF       1460 non-null   int64  
 15  LowQualFinSF   1460 non-null   int64  
 16  GrLivArea      1460 non-null  

#Here we are going to run the linear regression and for that we can't have missing values and thus, we are
doing imputation.All the concepts of imputations are important as learned in the past.
Here a simple imputation is done to keep on progressing with the feature selection lecture only
and this is not a guidance that only simple imputations should be done before feature selection

In [6]:
# Cell 3: Missing value imputation
# Purpose: make the regression model robust by filling missing values instead of letting them break the training process.
# Sub-part 1: loop through every column in the cleaned dataframe.
# Sub-part 2: if a column still contains missing values, compute its mean and replace the gaps with that mean.
# Sub-part 3: print the imputation result so the transformation is transparent and easy to audit.

# Step 4: Impute remaining null values

for col in df_cleaned.columns:
    if df_cleaned[col].isnull().sum() > 0:
        mean_val = df_cleaned[col].mean()
        # Fix the warning by reassigning the column
        df_cleaned[col] = df_cleaned[col].fillna(mean_val)
        print(f"Imputed missing values in '{col}' with the mean ({mean_val}).")
print("-" * 30)


Imputed missing values in 'LotFrontage' with the mean (70.04995836802665).
Imputed missing values in 'MasVnrArea' with the mean (103.68526170798899).
Imputed missing values in 'GarageYrBlt' with the mean (1978.5061638868744).
------------------------------


In [8]:
# Cell 4: Define the target and the feature matrix
# Purpose: separate the target variable from the predictors so the selection methods can operate on the right data.
# Sub-part 1: X holds all candidate input features.
# Sub-part 2: y holds the target variable that the model tries to predict.
# Important note: we exclude SalePrice-related columns from X to avoid using the target or its derived versions as features.

X = df_cleaned.drop(columns=["SalePrice"])
y = df_cleaned["SalePrice"]


In [9]:
X

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
0,1,60,65.0,8450,7,5,2003,2003,196.0,706,...,548,0,61,0,0,0,0,0,2,2008
1,2,20,80.0,9600,6,8,1976,1976,0.0,978,...,460,298,0,0,0,0,0,0,5,2007
2,3,60,68.0,11250,7,5,2001,2002,162.0,486,...,608,0,42,0,0,0,0,0,9,2008
3,4,70,60.0,9550,7,5,1915,1970,0.0,216,...,642,0,35,272,0,0,0,0,2,2006
4,5,60,84.0,14260,8,5,2000,2000,350.0,655,...,836,192,84,0,0,0,0,0,12,2008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,62.0,7917,6,5,1999,2000,0.0,0,...,460,0,40,0,0,0,0,0,8,2007
1456,1457,20,85.0,13175,6,6,1978,1988,119.0,790,...,500,349,0,0,0,0,0,0,2,2010
1457,1458,70,66.0,9042,7,9,1941,2006,0.0,275,...,252,0,60,0,0,0,0,2500,5,2010
1458,1459,20,68.0,9717,5,6,1950,1996,0.0,49,...,240,366,0,112,0,0,0,0,4,2010


In [11]:
y

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In regression analysis, adding a "constant column" (a column of all 1s) is how we introduce the **intercept** ($\beta_0$) into the model. Without it, you are forcing your model to make a very rigid, and often unrealistic, assumption about your data.

Here is a breakdown of why it’s statistically necessary, how it works, and when you should (or shouldn’t) include it.

## 1. The Statistical Need (With a Simple Example)

Mathematically, a simple linear regression equation looks like this:

$$
y = \beta_0 + \beta_1 x
$$

If you don't add a constant column, the matrix math used to solve for the coefficients can't calculate $\beta_0$. The equation effectively becomes:

$$
y = \beta_1 x
$$

### The Real-World Example: Predicting House Prices

Imagine you are predicting house prices ($y$) based on their square footage ($x$).

- **With a Constant ($\beta_0$):** Your model might look like $y = \$50,000 + \$150x$. This means even if a plot of land has 0 square feet of built area, the base value of the location/land itself is worth $\$50,000$. The line can cross the y-axis wherever the data dictates.
- **Without a Constant (No $\beta_0$):** Your model is forced to be $y = \beta_1 x$. This strictly forces the regression line to pass exactly through the origin $(0,0)$. In plain terms, it assumes that a 0 sq ft house costs exactly $\$0$.

By forcing the line through $(0,0)$, the model will tilt and warp the slope ($\beta_1$) just to hit that zero point, severely damaging its ability to accurately predict the actual data points you care about.

## 2. Pros and Cons of Adding a Constant

### Pros

- **Unbiased Predictions:** It ensures that the residual errors average out to exactly zero ($\sum e_i = 0$). Without an intercept, your model's predictions will be fundamentally biased (consistently over- or under-predicting).
- **Protects the Slope:** It allows the slope ($\beta_1$) to purely measure the *relationship* between the variables, rather than forcing the slope to double-duty as a corrector to drag the line down to the origin.
- **Validates Standard $R^2$:** Standard R-squared calculations assume an intercept is present. If you drop the constant, your software's $R^2$ output becomes misleadingly high or completely invalid.

### Cons

- **Extrapolation Trap:** The intercept represents the value of $y$ when all $x=0$. If $x=0$ is physically impossible or outside your data range (e.g., a person with 0 height or 0 weight), the intercept value itself becomes a meaningless "abstract" number.
- **Slight Loss of Degrees of Freedom:** Estimating $\beta_0$ consumes one degree of freedom, though this is negligible in almost all modern datasets.

## 3. When and When Not to Include a Constant

### When You SHOULD Include It (99% of the time)

- **Standard Practice:** Unless you have a bulletproof, theoretical reason not to, **always include it**.
- **When $x=0$ doesn't mean $y=0$:** For instance, predicting weight from height (a person with 0 height doesn't exist, and if they did, they wouldn't weigh 0 lbs in a linear trend).

### When You SHOULD NOT Include It

- **Strict Physical/Theoretical Constraints:** You should only drop the constant if it is a absolute certainty that when $x=0$, $y$ *must* be exactly $0$, and you want the model to enforce this.
- *Example:* Distance traveled ($y$) vs. Time spent driving ($x$) at a constant speed. If you have driven for 0 seconds, you have covered exactly 0 meters.
- **When using One-Hot Encoding for ALL categories:** If you convert a categorical variable (like "Color: Red, Blue, Green") into dummy variables and include *all* of them without dropping one, adding a constant column will cause perfect **multicollinearity** (the Dummy Variable Trap). In this specific scenario, you either drop one dummy category or drop the constant column.

*(Note: Most modern machine learning libraries, like `scikit-learn`'s `LinearRegression`, automatically include the constant/intercept by default so you don't have to manually add a column of 1s, whereas stats-heavy libraries like `statsmodels` require you to add it explicitly using `sm.add_constant(X)`).*

In [12]:
# Cell 5: Add an intercept term for regression
# Purpose: make the linear regression model capable of learning a baseline value when all predictors are zero.
# Sub-part 1: print the current shape of X before adding the intercept.
# Sub-part 2: add a constant column named const to X.
# Sub-part 3: print the updated shape and the new column names to confirm the transformation.

print(X.shape)
X = sm.add_constant(X) # column name will be 'const'
print(X.shape)
print(X.columns)


(1460, 37)
(1460, 38)
Index(['const', 'Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold'],
      dtype='str')


In [13]:
# Cell 6: Import a small utility module
# Purpose: keep this cell as a lightweight setup step before the iterative model loop begins.
# In this notebook, time is not used actively, but it is kept as a placeholder for possible timing or debugging work.

import time


When you run an OLS (Ordinary Least Squares) regression model in Python (like using the `statsmodels` library), it spits out a neat table filled with numbers, including a **p-value** for each of your features.

To understand how Python calculates this without drowning in heavy math, it helps to think of the OLS module as a **skeptical judge** running a simulation.

Here is the step-by-step breakdown of how Python determines that p-value.

## Step 1: Start with the "Skeptical" Assumption (The Null Hypothesis)

Before Python even looks at your data, it adopts a default mindset: **"This feature has absolutely zero effect on the outcome."** If you are using a person's *Shoe Size* to predict their *Salary*, the model starts by assuming the true relationship is exactly **0**. This baseline assumption is what statisticians call the **Null Hypothesis**.

## Step 2: Calculate the "Signal" (The Coefficient)

Next, Python looks at your actual data and draws the best-fitting regression line. It calculates the slope of that line, which is the **coefficient**.

Let's say Python looks at your data and finds that for every one-size increase in shoe size, salary goes up by **$5,000**.

- This $5,000 is your **Signal**. It’s what the data is telling us *might* be happening.

## Step 3: Calculate the "Noise" (The Standard Error)

Python knows that real-world data is messy. Maybe you just happened to have two outliers in your small dataset—a CEO with giant feet and a billionaire with tiny feet—that are throwing everything off.

So, Python calculates the **Standard Error**, which you can think of as the **Noise** or the uncertainty in your data.

- If your data points are tightly clustered around the line, the Noise is **low**.
- If your data points are wildly scattered all over the place, the Noise is **high**.

## Step 4: The T-Statistic (Signal divided by Noise)

Now, Python combines the Signal and the Noise into a single score called the **t-statistic**. The formula is conceptually very simple:

$$
\text{t-statistic} = \frac{\text{Signal}}{\text{Noise}}
$$

- **High t-statistic:** The signal is much louder than the noise. (e.g., $5,000 signal / $500 noise = 10).
- **Low t-statistic:** The noise is drowning out the signal. (e.g., $5,000 signal / $6,000 noise = 0.83).

## Step 5: The Final Calculation (The P-Value)

This is where the p-value is born. Python takes that **t-statistic** and compares it to a theoretical curve called the **t-distribution** (which looks like a standard bell curve).

Python asks this exact question:

> *"If the true relationship is actually ZERO (our Step 1 assumption), what are the chances that a random scribble of data would accidentally give me a t-statistic this large?"*
>
>

- **If the p-value is high (e.g., 0.40):** Python says, *"There is a 40% chance this result was just random noise. The judge remains skeptical. This feature is not statistically significant."*
- **If the p-value is low (e.g., 0.01):** Python says, *"There is only a 1% chance we would see a signal this strong purely by luck. The baseline assumption is broken. This feature is highly significant."*

### Summary in One Sentence

Python calculates the p-value by measuring **how many times louder your feature's signal is than the data's noise**, and checking how rare that loudness would be if the feature actually mattered zero percent of the time.

Choosing the right test statistic and distribution is essentially a game of **matching the rules of your data to the rules of mathematics**.

Python’s OLS module doesn't guess; it follows a strict, logical roadmap based on how the data was gathered and what we are trying to test. Here is how that decision is made, kept simple.

## 1. How Python Decides to Use the "T" Statistic

The choice of the **t-statistic** (instead of other statistics like a Z-score, F-statistic, or Chi-Square) comes down to two main factors: **what you are testing** and **what you know about the universe.**

- **The Goal:** You are testing a *single* specific feature's slope ($\beta_1$) to see if it is different from zero.
- **The Reality Check:** To use a standard $Z$-score (the classic normal distribution statistic), you have to know the *true, absolute variability of the entire population*. In the real world, we never know this. We only have our messy sample data.

Because we are **estimating** the noise (standard error) from our sample rather than knowing the true population noise perfectly, mathematical law dictates we must use the **t-statistic**.

*Small Dataset (e.g., 10 rows):** If you have very little data, Python uses a t-distribution with **"fat tails."** A fat tail means the curve is wider at the bottom. Why? Because with only 10 rows of data, random fluke results are highly likely. The curve makes it *harder* to get a low p-value because it gives the benefit of the doubt to randomness.
- **Large Dataset (e.g., 1,000 rows):** If you have a massive dataset, the t-distribution dynamically shifts. The tails get incredibly thin, and it morphs perfectly into a standard **Normal (Gaussian) Distribution**. With 1,000 rows, Python is confident that the noise is averaging out, so it doesn't need to be as forgiving to flukes.

## Summary of the OLS Cheat Sheet

When you run an OLS regression, Python follows this automatic checklist:

| The Question | Python's Logic | The Decision |
| --- | --- | --- |
| **What are we testing?** | A single feature's individual impact. | Use the **t-statistic**. |
| **Do we know the true population variance?** | No, we are estimating it from a sample. | Use the **t-distribution** curve. |
| **How much data do we have?** | Calculates $N - \text{features} - 1$. | Adjusts the **width/tails** of the t-distribution. |

### What about other statistics in the OLS table?

You might notice an **F-statistic** and an **F-probability** in the top right of your Python regression summary. Python switches to the **F-distribution** there because the question changes:

- **T-test question:** "Does *this specific feature* matter?" $\rightarrow$ **T-distribution**
- **F-test question:** "Do *all of these features combined* matter at all?" $\rightarrow$ **F-distribution**

In backward elimination, the p-value used to drop a feature takes into account all the other features currently in the model. It does not look at the feature in isolation.

Mathematically, it evaluates the controlled or conditional relationship between that individual feature and the target variable, given that the other features are also present

When you move from a simple regression (one $X$) to a multiple regression (multiple $X$'s predicting $Y$), the way Python calculates the p-value shifts from measuring a **simple relationship** to measuring a **unique contribution**.

Let’s strip away the matrix algebra and look at how Python calculates these simultaneous p-values using a visual analogy and a simple step-by-step concept.

## The Visual Analogy: Overlapping Venn Diagrams

Imagine you are trying to predict a student’s **Exam Score ($Y$)**. You have two features:

1. **Hours Studied ($X_1$)**
2. **Class Attendance ($X_2$)**

Naturally, students who attend class regularly also tend to study more. Their information overlaps.

If Python evaluated them in isolation, both would look incredibly powerful. But in a combined OLS model, Python does something clever: **it chops away the shared credit.**

- **The Total Information:** The entire circle of $Y$ is what we want to explain.
- **The Overlap:** The middle section where $X_1$ and $X_2$ overlap with each other *and* $Y$ is "shared credit." Python recognizes this overlap but refuses to give it to either feature individually when calculating their specific p-values.
- **The Unique Slice:** Python looks *only* at the slice of $Y$ that **Hours Studied ($X_1$)** explains *that nothing else in the model can explain*.

## Step-by-Step: How Python Calculates the Combined P-Value

Behind the scenes, the OLS module uses a mathematical trick called **partialling out** to calculate the p-value for multiple features simultaneously. Here is the 3-step process for how it calculates the p-value for just **Hours Studied ($X_1$)**:

### Step 1: Strip the "Shared Info" from the Feature

First, Python looks at **Hours Studied ($X_1$)** and **Attendance ($X_2$)**. It uses $X_2$ to predict $X_1$, and throws away the part they have in common. What is left over is the "pure, unique essence" of Hours Studied. Let's call this the **Unique Feature Signal**.

### Step 2: Strip the "Shared Info" from the Target

Next, Python looks at the **Exam Score ($Y$)**. It uses **Attendance ($X_2$)** to predict the score, and removes that chunk of the score from the equation. What is left over is the portion of the exam score that attendance couldn't explain. Let's call this the **Unexplained Target**.

### Step 3: Run a "Cleaned" T-Test

Finally, Python runs a standard t-test (Signal divided by Noise) comparing *only* the **Unique Feature Signal** against the **Unexplained Target**.

$$
\text{t-statistic for } X_1 = \frac{\text{Unique Signal of } X_1 \text{ left over after removing } X_2}{\text{Remaining Noise}}
$$

From this customized t-statistic, Python calculates the final p-value for $X_1$.

## Why this happens "Simultaneously"

Python doesn't just do this for one feature—it does this for **every single feature in the model at the exact same time.** When you look at the final OLS summary table:

- The p-value for $X_1$ assumes $X_2, X_3, \dots$ are held constant.
- The p-value for $X_2$ assumes $X_1, X_3, \dots$ are held constant.

This is why, in backward elimination, removing a feature causes all other p-values to change. If you fire $X_2$, the "shared credit" it was hoarding is suddenly up for grabs, and $X_1$ might instantly absorb it, causing $X_1$'s unique signal to get much stronger and its p-value to drop!

In [14]:
# Cell 7: Backward Elimination using p-values
# Purpose: start with all features and remove the least statistically significant ones until the remaining features are all useful enough.
# Sub-part 1: define the significance threshold at 0.05, which is a common cutoff in regression analysis.
# Sub-part 2: fit an OLS regression model on the current feature set.
# Sub-part 3: inspect the p-values of all predictors, excluding the intercept.
# Sub-part 4: if the highest p-value is above the threshold, remove that feature and repeat.
# Sub-part 5: stop the loop when all remaining predictors pass the significance test.

# H[0] assumption is that the feature is not significant (p-value > 0.05) and H[1] assumption is that the feature is significant (p-value <= 0.05)
# That is the reason we are removing the feature with the highest p-value if it is greater than 0.05.

threshold = 0.05
print("Performing Backward Elimination:")

model_summary_flag = True

while True:
    model = sm.OLS(y, X).fit()
    # OLS -> Ordinary least squares
    # print(model.params)
    p_values = model.pvalues.drop("const") # exclude the intercept from p-value consideration
    # p_values is a pandas Series with feature names as index and their corresponding p-values as values
    max_p = p_values.max()

    if not model_summary_flag:
        print(model.summary())
        model_summary_flag = True

    if max_p > threshold:
        excluded_feature = p_values.idxmax() # get the feature name with the highest p-value
        print(f"Removing {excluded_feature} with p-value {max_p:.4f}")
        X = X.drop(columns=[excluded_feature])
        # time.sleep(4)
    else:
        break

# print(model.summary())

selected_BE = X.columns.to_list()

print(f"Final features: ", X.columns.to_list())


Performing Backward Elimination:
Removing BsmtUnfSF with p-value 0.9638
Removing MoSold with p-value 0.8957
Removing BsmtFinSF2 with p-value 0.8651
Removing OpenPorchSF with p-value 0.8426
Removing LowQualFinSF with p-value 0.7041
Removing MiscVal with p-value 0.6921
Removing GarageArea with p-value 0.6554
Removing BsmtHalfBath with p-value 0.6304
Removing Id with p-value 0.6147
Removing 3SsnPorch with p-value 0.4956
Removing EnclosedPorch with p-value 0.4933
Removing HalfBath with p-value 0.4700
Removing LotFrontage with p-value 0.2841
Removing 2ndFlrSF with p-value 0.2566
Removing 1stFlrSF with p-value 0.7595
Removing YrSold with p-value 0.2556
Removing PoolArea with p-value 0.1832
Removing GarageYrBlt with p-value 0.1254
Final features:  ['const', 'MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'TotalBsmtSF', 'GrLivArea', 'BsmtFullBath', 'FullBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCa

Now that you have a solid grasp on Backward Elimination, let’s flip the script.

Welcome to **Forward Selection (FS)**. If Backward Elimination is a "reality show elimination," Forward Selection is like building a "dream team" from scratch, recruiting players one by one based on who adds the most value to the current roster.

## What is Forward Selection?

Forward Selection starts with an **empty model** (just the intercept, no features). You then evaluate every available feature individually, find the one that performs the best, and add it to the model. You repeat this process, adding one feature at a time, until no remaining features can significantly improve the model.

### The Step-by-Step Process:

1. **Choose a significance level:** Set a $p$-value threshold for entry (e.g., $\alpha = 0.05$).
2. **Test all individual features:** Fit a separate single-feature model for every available feature.
3. **Pick the best performer:** Select the feature with the lowest $p$-value. If its $p$-value is less than your significance level ($p \lt \alpha$), add it to the model.
4. **Re-evaluate the rest:** Now, test all remaining features *in combination with* the feature(s) already selected.
5. **Repeat or Stop:** Find the one with the lowest $p$-value among the remaining candidates. If its $p$-value is still $\lt \alpha$, add it and go back to Step 4. If no remaining feature has a $p$-value below the threshold, stop.

## The Hypotheses: What are we testing at each step?

Unlike Backward Elimination where we test if we can *discard* a feature, Forward Selection tests if a new feature is *worthy of being added* given the features we've already committed to.

At any given step, let's say we already have a subset of features in our model (we'll call this set $S$). We are testing a candidate feature, $X_k$, to see if we should add it.

- **The Null Hypothesis ($H_0$):** $\beta_k = 0$ > *"Adding the feature $X_k$ to the existing model $S$ does **not** significantly improve the model's predictive power. Its coefficient is statistically indistinguishable from zero."* > >
- **The Alternative Hypothesis ($H_1$):** $\beta_k \neq 0$ > *"Adding the feature $X_k$ to the existing model $S$ **does** significantly improve the model. Its coefficient is statistically different from zero."* > >

### Why this is a "Conditional" Hypothesis:

This is a **Partial $F$-test** (or a conditional $t$-test). The hypothesis is calculated *conditioned on what is already in the model*. For example, if you already have "Square Footage" in a house-pricing model, the hypothesis test for adding "Number of Bedrooms" is checking: *"Does bedrooms add any **new** predictive value that square footage hasn't already explained?"*

## When to Use Forward Selection

- **When you have high-dimensional data ($p \gt n$):** If you have 10,000 genes (features) but only 100 patients, you can't run Backward Elimination because you can't fit the starting "all-features" model. Forward Selection handles this easily because it starts with zero features.
- **When you expect only a few features to be useful:** If you have 500 features but suspect only 5 or 10 actually matter, Forward Selection is lightning-fast. It will stop after a few iterations rather than training massive models.
- **When computing resources are limited:** It is generally much lighter on memory early on compared to starting with a bloated, full-feature model.

## When NOT to Use It

- **When feature interactions are crucial:** This is Forward Selection's biggest blind spot. Suppose Feature A and Feature B are useless on their own, but *extremely* powerful when combined. Forward Selection will test Feature A (looks bad, gets rejected) and test Feature B (looks bad, gets rejected). It will never see their combined magic.
- **When features are highly correlated (Collinearity):** If Feature X and Feature Y are highly correlated, once Feature X is selected, Feature Y's $p$-value will skyrocket (because X already explains the variance). Y will be locked out forever, even if Y was actually the slightly better physical metric.

## Key Details & Pitfalls You Must Be Aware Of

- **The "Hotel California" Rule (You can never leave):** Just like features eliminated in BE are gone forever, features added in Forward Selection **can never be removed**. Even if adding Feature #4 makes Feature #1 completely redundant, Feature #1 stays in the model.
- **The Solution to this Pitfall (Stepwise Selection):** To fix this "greedy" lock-in, statisticians created **Bidirectional Elimination** (often just called **Stepwise Selection**). It is a hybrid: it adds a feature (like Forward Selection), but after each addition, it runs a backward elimination check to see if any previously added features have now become redundant and should be kicked out.


In [15]:
# Cell 8: Forward Selection using p-values
# Purpose: start with no features and add predictors incrementally when they are statistically significant.
# Sub-part 1: define a helper function that evaluates each candidate feature one at a time.
# Sub-part 2: for every remaining feature, fit a regression model with the currently selected features plus that feature.
# Sub-part 3: choose the feature with the lowest p-value and keep it if the p-value is below the threshold.
# Sub-part 4: stop when no remaining feature improves the model enough to meet the significance rule.

# 2. Forward Selection (Pvalue)
# Backward elimination starts with all features and removes the least useful ones.
# Forward selection does the opposite.
# It starts with no features, then adds them one by one if they improve the model significantly.

# Steps:
# Start with an empty model (only the intercept).
# For each remaining feature, fit a regression with that feature added.
# Choose the feature that gives the lowest p-value and is below the threshold (e.g., 0.05).
# Keep adding features one by one until no feature improves the model.

def forward_selection(X, y, threshold=0.05):
    selected_features = []
    remaining_features = list(X.columns)

    while remaining_features:
        pvals = {}
        for feature in remaining_features:
            model = sm.OLS(y, sm.add_constant(X[selected_features + [feature]])).fit()
            pvals[feature] = model.pvalues[feature]
        min_pval_feature = min(pvals, key=pvals.get)
        min_pval = pvals[min_pval_feature]
        if min_pval < threshold:
            selected_features.append(min_pval_feature)
            remaining_features.remove(min_pval_feature)
            print(f"Selected {min_pval_feature} with p-value {min_pval:.4f}")
        else:
            break
    return selected_features

X = df_cleaned.drop(columns=["SalePrice"])
y = df_cleaned["SalePrice"]

selected_FS = forward_selection(X, y)
print("Final features (Forward Selection):", selected_FS)


Selected OverallQual with p-value 0.0000
Selected GrLivArea with p-value 0.0000
Selected BsmtFinSF1 with p-value 0.0000
Selected GarageCars with p-value 0.0000
Selected MSSubClass with p-value 0.0000
Selected YearBuilt with p-value 0.0000
Selected BedroomAbvGr with p-value 0.0000
Selected OverallCond with p-value 0.0000
Selected LotArea with p-value 0.0000
Selected MasVnrArea with p-value 0.0000
Selected BsmtFullBath with p-value 0.0000
Selected TotRmsAbvGrd with p-value 0.0003
Selected WoodDeckSF with p-value 0.0015
Selected ScreenPorch with p-value 0.0007
Selected TotalBsmtSF with p-value 0.0054
Selected YearRemodAdd with p-value 0.0069
Selected KitchenAbvGr with p-value 0.0139
Selected Fireplaces with p-value 0.0354
Selected FullBath with p-value 0.0496
Final features (Forward Selection): ['OverallQual', 'GrLivArea', 'BsmtFinSF1', 'GarageCars', 'MSSubClass', 'YearBuilt', 'BedroomAbvGr', 'OverallCond', 'LotArea', 'MasVnrArea', 'BsmtFullBath', 'TotRmsAbvGrd', 'WoodDeckSF', 'ScreenPorc

In [16]:
# Cell 23: Compare Backward Elimination and Forward Selection results
# Purpose: find common and uncommon features between the two selection methods.
# Sub-part 1: remove 'const' from selected_BE since it's the intercept, not a feature.
# Sub-part 2: compute the intersection (common features) using set operations.
# Sub-part 3: compute the symmetric difference (uncommon features) in each method.
# Sub-part 4: display the results in a clear format.

# Remove 'const' from selected_BE for fair comparison
selected_BE_features = [f for f in selected_BE if f != 'const']

# Convert to sets for easier comparison
set_BE = set(selected_BE_features)
set_FS = set(selected_FS)

# Common features (intersection)
common_features = set_BE.intersection(set_FS)

# Uncommon features
only_in_BE = set_BE - set_FS
only_in_FS = set_FS - set_BE

print("=" * 60)
print("COMPARISON: Backward Elimination vs Forward Selection")
print("=" * 60)
print(f"\nCommon Features ({len(common_features)}):")
print(sorted(list(common_features)))

print(f"\nFeatures Only in Backward Elimination ({len(only_in_BE)}):")
print(sorted(list(only_in_BE)))

print(f"\nFeatures Only in Forward Selection ({len(only_in_FS)}):")
print(sorted(list(only_in_FS)))

print("\n" + "=" * 60)
print(f"Total in BE: {len(selected_BE_features)}")
print(f"Total in FS: {len(selected_FS)}")
print(f"Common: {len(common_features)}")
print("=" * 60)

COMPARISON: Backward Elimination vs Forward Selection

Common Features (19):
['BedroomAbvGr', 'BsmtFinSF1', 'BsmtFullBath', 'Fireplaces', 'FullBath', 'GarageCars', 'GrLivArea', 'KitchenAbvGr', 'LotArea', 'MSSubClass', 'MasVnrArea', 'OverallCond', 'OverallQual', 'ScreenPorch', 'TotRmsAbvGrd', 'TotalBsmtSF', 'WoodDeckSF', 'YearBuilt', 'YearRemodAdd']

Features Only in Backward Elimination (0):
[]

Features Only in Forward Selection (0):
[]

Total in BE: 19
Total in FS: 19
Common: 19


Using $p$-values for feature selection is deeply rooted in classical statistical inference. However, in machine learning, we often care more about **how well our model explains the data** than we do about strict probability thresholds.

This is where **Forward Selection using $R^2$ (specifically Adjusted $R^2$)** comes into play. Let's break down how this metric-driven approach works.

## 1. A Brief Primer: What is $R^2$ and Adjusted $R^2$?

Before using it for feature selection, we have to understand the math.

### $R^2$ (Coefficient of Determination)

$R^2$ measures the proportion of variance in the dependent variable (the target) that is predictable from the independent variables (the features).

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

- **$0$ (or $0\%$):** The model explains none of the variability.
- **$1$ (or $100\%$):** The model perfectly explains all the variability.

> ⚠️ **The Golden Rule of $R^2$:** Raw $R^2$ is mathematically guaranteed to **never decrease** when you add a new feature, even if you add a column of completely random numbers.
>
>

### Adjusted $R^2$ (The Savior)

Because raw $R^2$ is easily "cheated" by adding useless features, we use **Adjusted $R^2$** for feature selection. It penalizes the score for every feature ($k$) you add relative to your sample size ($n$):

$$
R^2_{adj} = 1 - \left[ \frac{(1 - R^2)(n - 1)}{n - k - 1} \right]
$$

If you add a useless feature, the penalty term outweighs any tiny, random improvement in $R^2$, causing the Adjusted $R^2$ to **drop**.

## 2. What is Forward Selection using $R^2$?

Instead of looking at $p$-values to decide which feature to add, we look at which feature gives the biggest boost to our **Adjusted $R^2$**.

### The Step-by-Step Process:

1. **Start empty:** Fit a model with no features (just the intercept). Compute its $R^2$ (which is 0).
2. **Test individually:** Fit a single-feature model for every candidate feature. Record the $R^2$ for each.
3. **Pick the best:** Add the feature that yields the **highest Adjusted $R^2$**.
4. **The Loop:** Try adding each remaining feature one by one alongside the already selected feature(s). Calculate the new Adjusted $R^2$ for each combination.
5. **Stopping Criterion:** Add the feature that gives the highest increase in Adjusted $R^2$. **Stop** the moment adding any new feature causes the Adjusted $R^2$ to decrease or flatten out (i.e., the penalty for complexity is larger than the predictive benefit).

## 3. The Hypotheses in this Case

When selecting features based on $R^2$, we are comparing a simpler model (without the new feature) to a more complex model (with the new feature). Technically, this maps to an **$F$-test for the change in $R^2$**.

For any candidate feature $X_k$ being considered for addition to the existing model:

- **Null Hypothesis ($H_0$):** $\Delta R^2_{adj} \le 0$ > *"The addition of feature $X_k$ does **not** increase the proportion of variance explained in the population (the Adjusted $R^2$ change is zero or negative once penalized for model complexity)."* > >
- **Alternative Hypothesis ($H_1$):** $\Delta R^2_{adj} \gt 0$ > *"The addition of feature $X_k$ **does** significantly increase the proportion of variance explained in the population (the improvement is strong enough to overcome the complexity penalty)."* > >

## 4. When to Use and When Not

### When to Use:

- **Your goal is pure prediction/explanation:** If you care about building a model that explains the maximum amount of variance in your data, Adjusted $R^2$ is your direct metric.
- **You want a practical stopping point:** Unlike arbitrary $p$-value thresholds (like $0.05$), Adjusted $R^2$ has a built-in mathematical stopping point: stop when the curve peaks and starts dropping.

### When NOT to Use:

- **You are doing strict scientific inference:** If your paper requires proving a feature has a statistically significant "effect" with a confidence interval, $p$-values are still the gold standard.
- **With very small datasets ($n$ is small):** Adjusted $R^2$ can still be highly volatile with tiny sample sizes, leading the algorithm to overfit to noise.

## 5. Key Nuances: $p$-value vs. $R^2$ Selection

While both methods often select similar features, their underlying philosophies differ significantly:

| Dimension | $p$-value Selection | Adjusted $R^2$ Selection |
| --- | --- | --- |
| **Primary Focus** | **Statistical Significance** (Is this relationship real or random noise?) | **Practical Significance** (How much actual predictive power does this add?) |
| **Stopping Rule** | Hard threshold (e.g., $p \ge 0.05$). | Natural peak (when Adjusted $R^2$ starts to decrease). |
| **Sensitivity to Sample Size** | **Extremely sensitive.** In huge datasets ($n = 1,000,000$), even useless features will have a $p$-value of $\lt 0.001$. | Less sensitive. It scales with $n$ and focuses on the percentage of variance explained, preventing minor effects from dominating. |
| **Risk** | Can select features that are statistically real but practically useless (explaining $0.0001\%$ of variance). | Can accidentally include marginally noisy features if the penalty threshold isn't strict enough. |

### The "Big Picture" Takeaway

- Use **$p$-values** if you want to be conservative and prove that every single variable in your model has a statistically verifiable, non-zero relationship with the target.
- Use **Adjusted $R^2$** if you want to maximize your model's predictive punch and don't want to get bogged down in standard hypothesis testing rules.

In [19]:
# Cell 12: Forward Selection using R-squared
# Purpose: try a different selection strategy that uses model fit instead of p-values.
# Sub-part 1: define a helper function that evaluates each remaining feature by fitting a regression model.
# Sub-part 2: compute the R-squared of each candidate model and pick the one that improves the fit the most.
# Sub-part 3: keep adding features until no further improvement in R-squared is observed.

# 2. Forward Selection (Pvalue)
def forward_selection_r_squared(X, y):
    selected_features = []
    remaining_features = list(X.columns)
    best_r_squared = -1

    while remaining_features:
        best_candidate = None
        current_best_r_squared = -1  # BAD

        for feature in remaining_features:
            model_features = selected_features + [feature]
            model = sm.OLS(y, sm.add_constant(X[model_features])).fit()

            # Check if this model improves R-squared
            if model.rsquared > current_best_r_squared:
                current_best_r_squared = model.rsquared
                best_candidate = feature

        if current_best_r_squared > best_r_squared:
            selected_features.append(best_candidate)
            remaining_features.remove(best_candidate)
            best_r_squared = current_best_r_squared
            print(f"Selected {best_candidate} with R-squared {best_r_squared:.4f}")
        else:
            break

    return selected_features

X = df_cleaned.drop(columns=["SalePrice"])
y = df_cleaned["SalePrice"]

selected_FS_R2 = forward_selection_r_squared(X, y)
print("Final features (Forward Selection):", selected_FS_R2)


Selected OverallQual with R-squared 0.6257
Selected GrLivArea with R-squared 0.7142
Selected BsmtFinSF1 with R-squared 0.7459
Selected GarageCars with R-squared 0.7662
Selected MSSubClass with R-squared 0.7772
Selected YearBuilt with R-squared 0.7852
Selected BedroomAbvGr with R-squared 0.7898
Selected OverallCond with R-squared 0.7941
Selected LotArea with R-squared 0.7977
Selected MasVnrArea with R-squared 0.8011
Selected BsmtFullBath with R-squared 0.8033
Selected TotRmsAbvGrd with R-squared 0.8051
Selected WoodDeckSF with R-squared 0.8064
Selected ScreenPorch with R-squared 0.8079
Selected TotalBsmtSF with R-squared 0.8090
Selected YearRemodAdd with R-squared 0.8099
Selected KitchenAbvGr with R-squared 0.8107
Selected Fireplaces with R-squared 0.8113
Selected FullBath with R-squared 0.8118
Selected GarageYrBlt with R-squared 0.8121
Selected PoolArea with R-squared 0.8123
Selected YrSold with R-squared 0.8125
Selected LowQualFinSF with R-squared 0.8127
Selected LotFrontage with R-sq

In [20]:
# Cell 13: Compare R-squared-based selection with p-value-based selection
# Purpose: check whether both forward-selection variants agree on the same features.
# This helps illustrate that different scoring rules can produce different feature sets even for the same problem.

common = set(selected_FS_R2).intersection(selected_FS)
print(common)

print(len(selected_FS_R2))
print(len(selected_FS))
print(len(common))


{'BsmtFinSF1', 'MasVnrArea', 'WoodDeckSF', 'OverallQual', 'KitchenAbvGr', 'FullBath', 'YearBuilt', 'ScreenPorch', 'GarageCars', 'YearRemodAdd', 'MSSubClass', 'GrLivArea', 'TotalBsmtSF', 'LotArea', 'BsmtFullBath', 'TotRmsAbvGrd', 'Fireplaces', 'BedroomAbvGr', 'OverallCond'}
35
19
19


We’ve covered building up (Forward Selection) and tearing down (Backward Elimination) using classical statistics. Now, let's step squarely into modern machine learning with **Recursive Feature Elimination (RFE)**.

RFE bridges the gap between statistical filtering and pure predictive modeling.

## What is Recursive Feature Elimination (RFE)?

RFE is a **backward selection wrapper method**, but instead of relying on $p$-values or $R^2$ from a statistical summary, it relies directly on the **feature importances** or **coefficients** generated by a machine learning algorithm (like Support Vector Machines, Random Forests, or Linear Regression).

### The Step-by-Step Process:

1. **Train the Model:** Fit your chosen machine learning estimator on the entire dataset using all features.
2. **Compute Importance:** The model ranks each feature based on its internal metrics (e.g., `.coef_` for linear models or `.feature_importances_` for tree-based models).
3. **Prune the Weakest:** Remove the feature(s) with the lowest importance score.
4. **Recalculate:** Re-train the model on the remaining features. (This step is crucial because when you drop one feature, the importance scores of the *remaining* features shift).
5. **Stop:** Repeat steps 2–4 until you reach a user-defined target number of features.

## The Statistical / Machine Learning Concept Behind It

RFE does not look at independent probability distributions ($p$-values). Instead, it relies on the concept of **Model-Driven Feature Attribution**.

When a machine learning model minimizes a loss function (like Mean Squared Error or Cross-Entropy), it assigns weights to features.

- In a linear model ($y = \beta_1x_1 + \beta_2x_2$), a coefficient $\beta_i$ close to $0$ means that changing $x_i$ barely changes the prediction.
- In a tree-based model, a low "Gini importance" means a feature is rarely chosen to split the data, or its splits don't reduce impurity very much.

RFE uses these internal model metrics as a proxy for feature value.

## What are the Hypotheses in RFE?

Strictly speaking, **RFE is an algorithmic optimization technique, not a statistical hypothesis test.** It does not naturally generate $p$-values, confidence intervals, or formal null hypotheses.

However, if we map the *intent* of RFE to a hypothesis framework, it looks like this:

- **Null Hypothesis ($H_0$):** $I(X_k) = 0$ > *"The machine learning model's internal importance metric ($I$) for feature $X_k$ is zero or the lowest among the remaining features, meaning it contributes no unique utility to minimizing the model's loss function."* > >
- **Alternative Hypothesis ($H_1$):** $I(X_k) \gt 0$ > *"The feature $X_k$ possesses a high enough importance rank that removing it would degrade the model's ability to minimize its loss function."* > >

## When to Use and When Not

### When to Use:

- **With Complex Machine Learning Models:** If you are planning to use a Random Forest, XGBoost, or an SVM, RFE uses the exact same model logic to select the features.
- **To Capture Non-linear Interactions:** Unlike $p$-value selection (which is usually tied to linear models), if you pair RFE with a Random Forest, it can successfully keep features that are highly important due to complex, non-linear interactions.
- **When You Have a Targeted Roster Size:** If your deployment constraints dictate, *"Our application can only handle exactly 10 inputs,"* RFE will elegantly prune down to exactly 10.

### When NOT to Use:

- **Computationally Brutal:** Because it retrains the model every time a feature is dropped, running RFE with 500 features on a slow-training model (like a deep neural network or a huge Random Forest) will take a very long time.
- **Uninterpretable Algorithms:** If you use RFE with a highly complex model, it will pick the best features, but it won't be able to tell you *why* or *how* those features relate to the target in a simple, linear way.

## RFE vs. Backward Elimination vs. Forward Selection (using $p$-values)

Here is how these three heavyweights stack up against each other:

| Dimension | Forward Selection ($p$-value) | Backward Elimination ($p$-value) | Recursive Feature Elimination (RFE) |
| --- | --- | --- | --- |
| **Starting Point** | Empty model (0 features) | Full model (All features) | Full model (All features) |
| **Core Metric** | Lowest $p$-value ($p \lt \alpha$) | Highest $p$-value ($p \gt \alpha$) | Model Importances / Coefficients |
| **Stopping Rule** | When no remaining feature passes the $p$-value threshold. | When all remaining features pass the $p$-value threshold. | When a pre-set **number** of features is reached (or optimized via cross-validation). |
| **Model Scope** | Limited mostly to Linear/Logistic Regression. | Limited mostly to Linear/Logistic Regression. | **Any** ML model that provides coefficients or feature importances. |
| **Handling Interdependence** | **Poor.** Misses features that only work well in pairs. | **Good.** Starts with all features to see how they behave together. | **Excellent.** Dynamically re-evaluates feature importance as others are removed. |
| **Speed** | Fast if only a few features are relevant. | Slow if there are many features. | Very slow (requires complete model retraining at every step). |

### Pro-Tip: RFE-CV

Because choosing the "target number of features" in RFE can feel like guesswork, data scientists almost always use **RFECV** (Recursive Feature Elimination with Cross-Validation). This variant runs RFE inside a cross-validation loop and automatically determines the *exact number* of features that maximizes your model's test score.

In [21]:
# Cell 14: Recursive Feature Elimination (RFE)
# Purpose: use a model-based approach to rank features and remove the least important ones iteratively.
# Sub-part 1: import RFE from sklearn, which performs the selection loop.
# Sub-part 2: use LinearRegression as the estimator to determine feature importance through model coefficients., not necessary to use LinearRegression, can use others as well.
# Sub-part 3: set n_features_to_select=10, meaning the algorithm should keep only 10 features in the end.
# Sub-part 4: fit the RFE object to the data so the ranking is computed.

# 3. Recursive Feature Elimination (RFE)

# RFE is model-based feature selection (unlike pure statistics with p-values).
# We fit a model (like Linear Regression, Logistic Regression, or Random Forest).
# Rank features by importance.
# Remove the least important feature(s).
# Repeat until the desired number of features remain.
# RFE works like backward elimination, but instead of p-values, it uses model coefficients or feature importance.
# Works with linear models, trees, etc. (Random Forest is often preferred for non-linear problems).

from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

X = df_cleaned.drop(columns=["SalePrice"])
y = df_cleaned["SalePrice"]

# Fit Linear Regression
model = LinearRegression()
# model = RandomForestRegressor(n_estimators=100, random_state=42)  # machine learning algo

rfe = RFE(model, n_features_to_select=10)
rfe = rfe.fit(X, y)


In [22]:
# Cell 15: Inspect the selected mask from RFE
# Purpose: check which features were retained by the algorithm and which ones were removed.
# The support_ array is a boolean mask where True means the feature was selected.

rfe.support_


array([False, False, False, False,  True, False, False, False, False,
       False, False, False, False, False, False, False, False,  True,
        True,  True,  True,  True,  True,  True,  True, False,  True,
       False, False, False, False, False, False, False, False, False,
       False])

In [23]:
# Cell 16: Inspect the ranking produced by RFE
# Purpose: understand the elimination order of the features.
# Sub-part 1: a rank of 1 indicates that the feature was selected and kept.
# Sub-part 2: higher ranks indicate features that were removed earlier because they were judged less important.
# Sub-part 3: the ranking order is useful for explaining which features were considered weakest.

rfe.ranking_

# 1 : important features and selected
# 2,3,4 : not selected (The higher the number, the less important)
# order defines the order in which they are eliminated


array([24,  5, 18, 27,  1,  2,  4,  7, 12, 19, 25, 28, 17,  9, 10, 21, 14,
        1,  1,  1,  1,  1,  1,  1,  1,  8,  1, 22, 13, 23, 20, 16, 11, 15,
       26,  6,  3])

In [24]:
# Cell 17: Extract the final features kept by RFE
# Purpose: convert the boolean mask into a readable list of selected feature names.
# This gives the final set of predictors that RFE recommends for the model.

selected_features_rfe = X.columns[rfe.support_]
print("Selected Features (RFE):", list(selected_features_rfe))


Selected Features (RFE): ['OverallQual', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars']


In [25]:
# Cell 18: Count the number of selected features
# Purpose: verify how many predictors remain after the RFE process.
# This is useful for checking whether the model has been reduced to the expected size.

len(selected_features_rfe)


10